# 10 — Linear Regression Assumptions

**Difficulty:** ⭐⭐⭐⭐ &nbsp;|&nbsp; **Estimated time:** 2.5 hours  
**Week 5 — ML Fundamentals & Linear Regression**

---

## Why This Matters

Linear regression is one of the most studied and deployed models in statistics and machine learning.  
But it comes with **six core assumptions**.  

Violating those assumptions does not always break your predictions — but it breaks the **mathematical guarantees** the model was built on:

- Coefficient estimates may become **biased**
- Standard errors may be **wrong**, making p-values unreliable
- Confidence intervals may be **too narrow** (you are more uncertain than the model admits)
- The model may be **unstable** — small changes in data produce large changes in coefficients

---

## Real-World Analogy First

**The Airbag Analogy**

A car's airbag system is engineered with precise assumptions:  
it deploys in a side-impact crash above 15 mph, on a paved road, with a standard crash profile.  

Inside those conditions → the airbag works perfectly and saves lives.  
Drive underwater, flip the car upside-down, or crash at 2 mph → the conditions are violated → the airbag may not deploy.

**Linear regression is exactly like this.** Under the right conditions (the assumptions), OLS is *provably optimal*.  
Outside those conditions, you need a different tool.

---

## The Gauss-Markov Theorem

The Gauss-Markov theorem is the theoretical backbone of linear regression:

> **If assumptions L, I, E (and linearity in parameters) hold, the OLS estimator is BLUE:**  
> **B**est **L**inear **U**nbiased **E**stimator

Breaking this down:
- **Best:** minimum variance among all linear unbiased estimators (you cannot do better with any linear method)
- **Linear:** applies only to linear estimators — a neural network is not in this competition
- **Unbiased:** $E[\hat{\theta}] = \theta^*$ — on average, OLS gives the true coefficient values
- **Estimator:** the OLS formula $\hat{\theta} = (X^TX)^{-1}X^Ty$

**The 6 Assumptions (remembered as LINE + NC):**

| Letter | Assumption |
|--------|------------|
| **L** | **L**inearity — relationship between X and y is linear |
| **I** | **I**ndependence — errors are independent of each other |
| **N** | **N**ormality — errors are normally distributed |
| **E** | **E**qual variance (homoscedasticity) — errors have constant variance |
| **N** | **N**o multicollinearity — features are not perfectly correlated |
| **C** | **C**ook's distance — no highly influential outliers |

---

**Dataset used throughout:** California Housing (sklearn)  
**Target:** log(MedHouseVal) — log of median house value

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
sns.set_palette('muted')
print('Libraries loaded.')

In [ ]:
# ── Load and Prepare Data ───────────────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Add engineered columns we will use throughout this notebook:
# 1. Log price — standard transformation for right-skewed price data
df['LogPrice'] = np.log(df['MedHouseVal'])

# 2. Simulated total_rooms and total_bedrooms (highly correlated)
#    In the real dataset, AveRooms is already per household.
#    We recreate the raw totals to demonstrate multicollinearity.
df['TotalRooms']    = df['AveRooms'] * df['Population']
df['TotalBedrooms'] = df['AveBedrms'] * df['Population']

# 3. Simulated time index (houses listed sequentially by district number)
#    California housing is cross-sectional, but we add a fake time index
#    so we can demonstrate autocorrelation
df['TimeIndex'] = np.arange(len(df))

print('Columns:', df.columns.tolist())
print('Shape  :', df.shape)
print(df[['MedHouseVal', 'LogPrice', 'TotalRooms', 'TotalBedrooms']].describe().round(2))

In [ ]:
# ── Baseline Feature Set and Train/Test Split ───────────────────────────────
# Standard features (excluding our engineered collinear ones for now)
base_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
                 'Population', 'AveOccup', 'Latitude', 'Longitude']

X = df[base_features]
y = df['LogPrice']      # log-transformed price as baseline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_sc, y_train)

y_pred    = model.predict(X_test_sc)
residuals = y_test.values - y_pred

print(f'Baseline R² (log price): {model.score(X_test_sc, y_test):.4f}')
print(f'Mean residual: {residuals.mean():.6f} (≈ 0 by construction)')

---

## Assumption L — Linearity

### What It Means

The relationship between each feature $X_j$ and the target $y$ is linear — a one-unit increase in $X_j$ produces a constant change in $y$ regardless of the current value of $X_j$.

**Analogy:** A linear model assumes that adding one extra bedroom always increases house price by the same dollar amount — whether the house goes from 1 to 2 bedrooms, or from 8 to 9 bedrooms.  
In reality, the first extra bedroom matters much more than the 9th.

### Mathematical Statement

$$y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_p X_p + \varepsilon$$

The relationship is linear **in the parameters** $\beta_j$ — not necessarily linear in the raw features.  
(A polynomial regression $y = \beta_0 + \beta_1 X + \beta_2 X^2$ is still linear in $\beta$, just not in $X$.)

### How to Check

1. Scatter plot of each feature vs $y$ (look for curves)
2. Residuals vs fitted values (should be a horizontal cloud)
3. Residuals vs each feature (LOWESS should be flat)

In [ ]:
# ── L: Linearity Violation + Fix ────────────────────────────────────────────
# HouseAge vs LogPrice has a non-linear (saturating) relationship
# New houses are cheap (condos); middle-aged are desirable; very old can be either

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: Raw scatter with linear fit
ax = axes[0]
ax.scatter(df['HouseAge'], df['LogPrice'], alpha=0.05, s=5, color='steelblue')
m_lin = np.polyfit(df['HouseAge'], df['LogPrice'], 1)
x_lin = np.linspace(df['HouseAge'].min(), df['HouseAge'].max(), 100)
ax.plot(x_lin, np.polyval(m_lin, x_lin), 'r-', linewidth=2, label='Linear fit')
ax.set_xlabel('House Age (years)')
ax.set_ylabel('Log Price')
ax.set_title('HouseAge vs LogPrice\n(Possibly non-linear at extremes)')
ax.legend()

# Panel 2: Residuals vs HouseAge — detect the non-linear pattern
ax = axes[1]
age_arr = X_test['HouseAge'].values
ax.scatter(age_arr, residuals, alpha=0.2, s=8, color='steelblue')
ax.axhline(0, color='red', linestyle='--')
sort_idx = np.argsort(age_arr)
lowess_out = sm.nonparametric.lowess(residuals[sort_idx], age_arr[sort_idx], frac=0.4)
ax.plot(lowess_out[:, 0], lowess_out[:, 1], 'r-', linewidth=2, label='LOWESS trend')
ax.set_xlabel('House Age')
ax.set_ylabel('Residual')
ax.set_title('Residuals vs HouseAge\n(Curve = missing non-linear term)')
ax.legend()

# Panel 3: Fix — add polynomial term HouseAge²
ax = axes[2]
X_poly_train = X_train[['HouseAge']].copy()
X_poly_train['HouseAge2'] = X_poly_train['HouseAge'] ** 2
X_poly_test  = X_test[['HouseAge']].copy()
X_poly_test['HouseAge2']  = X_poly_test['HouseAge'] ** 2

# Fit a curve to show the polynomial captures the pattern better
m_poly = np.polyfit(df['HouseAge'], df['LogPrice'], 2)   # degree-2 polynomial
ax.scatter(df['HouseAge'], df['LogPrice'], alpha=0.05, s=5, color='seagreen')
ax.plot(x_lin, np.polyval(m_poly, x_lin), 'r-', linewidth=2,
        label='Quadratic fit (HouseAge²)')
ax.set_xlabel('House Age (years)')
ax.set_ylabel('Log Price')
ax.set_title('Fix: Add HouseAge² Term\n(Captures curvature)')
ax.legend()

plt.suptitle('Assumption L — Linearity: Violation & Fix',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Assumption I — Independence of Errors

### What It Means

The error for observation $i$ should carry no information about the error for observation $j$.

$$\text{Cov}(\varepsilon_i, \varepsilon_j) = 0 \quad \text{for all } i \neq j$$

**Analogy:** If you flip a fair coin, the result of flip 1 tells you nothing about flip 2.  
But if a biased coin starts landing heads, it tends to keep landing heads.  
Autocorrelated residuals are like a biased coin — the errors are "streaky."

### Common Violations in House Price Data

1. **Time series:** prices listed on Monday cluster near Sunday's prices
2. **Spatial clustering:** houses in the same neighborhood have correlated errors — location effects not captured by Latitude/Longitude alone
3. **Panel data:** the same house measured 12 months in a row → the errors across months are correlated

### Consequence of Violation

Standard errors are **under-estimated** → t-statistics are inflated → you declare features significant when they are not → over-confidence in coefficients.

In [ ]:
# ── I: Independence — Simulate and Detect Autocorrelation ──────────────────
# Sort the data by Latitude (as a proxy for spatial ordering)
# Houses sorted N→S should show spatial clustering in residuals

df_sorted = df.sort_values('Latitude').reset_index(drop=True)
X_sorted  = df_sorted[base_features]
y_sorted  = df_sorted['LogPrice']

X_sc_sorted = scaler.fit_transform(X_sorted)
model_sorted = LinearRegression().fit(X_sc_sorted, y_sorted)
resid_sorted = y_sorted.values - model_sorted.predict(X_sc_sorted)

dw_stat = durbin_watson(resid_sorted)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── Residuals vs index (spatial order) ────────────────────────────────────
ax = axes[0]
sample = np.arange(0, 2000)     # show first 2000 for clarity
ax.plot(sample, resid_sorted[sample], alpha=0.6, linewidth=0.6, color='steelblue')
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Index (houses sorted by Latitude: N → S)')
ax.set_ylabel('Residual')
ax.set_title(f'Residuals vs Spatial Index\nDurbin-Watson = {dw_stat:.3f}'
             f'  (2 = independent)')

# ── Autocorrelation function (ACF) of residuals ────────────────────────────
ax = axes[1]
n_lags = 30
acf_vals = [np.corrcoef(resid_sorted[:-lag], resid_sorted[lag:])[0, 1]
            if lag > 0 else 1.0
            for lag in range(n_lags + 1)]

# Confidence bands: ±1.96/√n marks the 95% significance threshold
conf_band = 1.96 / np.sqrt(len(resid_sorted))

ax.bar(range(n_lags + 1), acf_vals, color='steelblue', alpha=0.7)
ax.axhline(conf_band,  color='red', linestyle='--', linewidth=1.2,
           label=f'95% CI (±{conf_band:.3f})')
ax.axhline(-conf_band, color='red', linestyle='--', linewidth=1.2)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.set_title('ACF of Residuals\n(Bars outside red = autocorrelation present)')
ax.legend()

plt.suptitle('Assumption I — Independence: Spatial Autocorrelation',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Durbin-Watson statistic: {dw_stat:.4f}')
print('Fix: Add neighborhood dummies, use GLS with spatial covariance structure.')

---

## Assumption N — Normality of Errors

### What It Means

$$\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$$

Errors are normally distributed with mean zero and constant variance.

**Analogy:** Prediction errors should behave like static on a radio — equally likely to be a little too loud or too soft, rarely extreme, and centred on silence.

### Important Nuance

- **For predictions:** normality is NOT required. OLS gives good predictions without it.
- **For inference (p-values, confidence intervals):** normality IS required for exact tests.
- **Large samples (n > 100):** Central Limit Theorem partially rescues us — the sampling distribution of $\hat{\beta}$ approaches normal even if $\varepsilon$ is not perfectly normal.
- **Small samples:** normality matters a lot.

### Shapiro-Wilk Test

$H_0$: residuals are normally distributed  
$H_1$: residuals are not normally distributed  

Note: with n > 5000, even tiny harmless deviations will be detected. Always pair statistical tests with visual Q-Q inspection.

In [ ]:
# ── N: Normality — Before and After Log Transform ──────────────────────────
# Compare residuals from: (a) raw price OLS vs (b) log price OLS

y_raw = df['MedHouseVal']
X_all = df[base_features]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_all, y_raw, test_size=0.2, random_state=42
)
sc2 = StandardScaler()
Xtr_sc = sc2.fit_transform(X_train_r)
Xte_sc = sc2.transform(X_test_r)

m_raw = LinearRegression().fit(Xtr_sc, y_train_r)
resid_raw = y_test_r.values - m_raw.predict(Xte_sc)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for col_idx, (resid_arr, label, color) in enumerate([
    (resid_raw, 'Raw Price', 'steelblue'),
    (residuals, 'Log Price', 'seagreen')
]):
    # ─ Histogram row ────────────────────────────────────────────────────────
    ax = axes[0, col_idx]
    ax.hist(resid_arr, bins=60, density=True, color=color,
            alpha=0.7, edgecolor='white')
    x_g = np.linspace(resid_arr.min(), resid_arr.max(), 200)
    ax.plot(x_g,
            stats.norm.pdf(x_g, resid_arr.mean(), resid_arr.std()),
            'r-', linewidth=2, label='Normal curve')
    sk = stats.skew(resid_arr)
    ax.set_title(f'{label}: Histogram\nskewness = {sk:.3f}')
    ax.set_xlabel('Residual')
    ax.set_ylabel('Density')
    ax.legend()

    # ─ Q-Q row ──────────────────────────────────────────────────────────────
    ax = axes[1, col_idx]
    (osm, osr), (sl, ic, _) = stats.probplot(resid_arr, dist='norm')
    ax.scatter(osm, osr, alpha=0.3, s=8, color=color)
    ax.plot(osm, np.array(osm) * sl + ic, 'r-', linewidth=2)
    ax.set_title(f'{label}: Q-Q Plot')
    ax.set_xlabel('Theoretical Quantiles')
    ax.set_ylabel('Sample Quantiles')

plt.suptitle('Assumption N — Normality of Errors: Raw vs Log Price',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Shapiro-Wilk on a 2000-point sample (valid range for Shapiro)
rng = np.random.default_rng(0)
idx2k = rng.choice(len(resid_raw), size=2000, replace=False)
sw_raw = stats.shapiro(resid_raw[idx2k])
sw_log = stats.shapiro(residuals[idx2k])
print(f'Shapiro-Wilk — Raw price:  W={sw_raw.statistic:.4f}, p={sw_raw.pvalue:.6f}')
print(f'Shapiro-Wilk — Log price:  W={sw_log.statistic:.4f}, p={sw_log.pvalue:.6f}')

---

## Assumption E — Equal Variance (Homoscedasticity)

### What It Means

$$\text{Var}(\varepsilon_i) = \sigma^2 \quad \forall i$$

All errors have the same variance — the spread of errors is the same for cheap and expensive houses.

**Analogy:** A measuring tape should have the same precision for all lengths. If the tape gets wobblier as you extend it further, your measurements are **heteroscedastic** — less reliable for large objects.

### Why It Matters

When heteroscedasticity is present, OLS still produces **unbiased** estimates, but no longer **minimum variance** — weighted least squares (WLS) would be more efficient.  
More critically: the standard errors are wrong, making all confidence intervals and p-values unreliable.

### Scale-Location Plot

Plot $\sqrt{|\text{standardized residuals}|}$ vs fitted values.  
- **Flat horizontal band:** homoscedastic — all good  
- **Upward trend:** variance grows with predicted value — heteroscedastic  
- **Downward trend:** variance shrinks with predicted value (unusual)

In [ ]:
# ── E: Homoscedasticity — Scale-Location Plot + Breusch-Pagan ─────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (resid_arr, pred_arr, label, color) in zip(axes, [
    (resid_raw, m_raw.predict(Xte_sc), 'Raw Price', 'steelblue'),
    (residuals, y_pred, 'Log Price',   'seagreen')
]):
    std_resid_arr = (resid_arr - resid_arr.mean()) / resid_arr.std()
    sqrt_std      = np.sqrt(np.abs(std_resid_arr))

    ax.scatter(pred_arr, sqrt_std, alpha=0.2, s=8, color=color)
    lw = sm.nonparametric.lowess(sqrt_std, pred_arr, frac=0.35)
    ax.plot(lw[:, 0], lw[:, 1], 'r-', linewidth=2, label='LOWESS trend')

    # Breusch-Pagan test
    exog_bp = sm.add_constant(pred_arr)
    bp_lm, bp_p, _, _ = het_breuschpagan(resid_arr, exog_bp)

    ax.set_xlabel('Fitted Values')
    ax.set_ylabel('√|Standardized Residuals|')
    ax.set_title(f'Scale-Location — {label}\n'
                 f'Breusch-Pagan p={bp_p:.4f}'
                 f'  ({"Heteroscedastic" if bp_p < 0.05 else "Homoscedastic"})')
    ax.legend()

plt.suptitle('Assumption E — Equal Variance (Homoscedasticity)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Assumption N(2) — No Multicollinearity

### What It Means

Features should not be **perfectly (or near-perfectly) correlated** with each other.

**Why it matters — the math:**  
OLS computes $(X^TX)^{-1}$ to find the coefficients.  
If two columns of $X$ are perfectly correlated, $X^TX$ is **singular** (not invertible) — OLS fails completely.  
If they are near-perfectly correlated, $(X^TX)^{-1}$ has **huge entries** → coefficient estimates have huge variance → any small change in data leads to wildly different coefficients.

**Analogy:** Imagine asking someone "How much does the monthly rent affect price?" and "How much does the annual rent affect price?" at the same time. Monthly rent × 12 = annual rent — they carry identical information. The model cannot disentangle their individual effects.

### Variance Inflation Factor (VIF)

For each feature $j$, regress $X_j$ on all other features to get $R^2_j$.  
VIF quantifies how much the variance of coefficient $\hat{\beta}_j$ is **inflated** due to correlation:

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

| VIF | Interpretation |
|-----|----------------|
| 1 | No multicollinearity |
| 1–5 | Moderate — usually acceptable |
| 5–10 | High — worth investigating |
| > 10 | Severe — action needed |

In [ ]:
# ── N(2): Multicollinearity — Create Correlated Features + Compute VIF ─────
# We add TotalRooms and TotalBedrooms — these are highly correlated because
# TotalBedrooms ≈ 0.2 × TotalRooms (bedrooms are a fixed fraction of rooms)

multi_features = base_features + ['TotalRooms', 'TotalBedrooms']
X_multi = df[multi_features].copy()

# ── Correlation heatmap ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
corr = X_multi.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))   # upper triangle mask
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix\n(TotalRooms–TotalBedrooms near 1.0 = multicollinearity)')

# ── Compute VIF manually ───────────────────────────────────────────────────
def compute_vif(X_df):
    """Compute VIF for each column of X_df.
    
    VIF_j = 1 / (1 - R²_j), where R²_j is obtained by regressing
    column j on all other columns.
    """
    vif_data = {}
    cols = X_df.columns.tolist()
    X_arr = X_df.values.astype(float)

    for j, col in enumerate(cols):
        y_j   = X_arr[:, j]                          # target: feature j
        X_j   = np.delete(X_arr, j, axis=1)          # predictors: all other features
        model_j = LinearRegression().fit(X_j, y_j)
        r2_j  = model_j.score(X_j, y_j)              # R² of the regression
        # Guard against perfect fit causing 1/(1-1) = division by zero
        vif_j = 1 / (1 - r2_j) if r2_j < 1.0 else np.inf
        vif_data[col] = round(vif_j, 2)

    return pd.Series(vif_data, name='VIF').sort_values(ascending=False)

vif_series = compute_vif(X_multi)

# ── VIF bar chart ──────────────────────────────────────────────────────────
ax = axes[1]
colors_vif = ['crimson' if v > 10 else ('orange' if v > 5 else 'steelblue')
              for v in vif_series.values]
ax.barh(vif_series.index, vif_series.values, color=colors_vif, alpha=0.8)
ax.axvline(10, color='red', linestyle='--', linewidth=1.5, label='VIF = 10 (severe)')
ax.axvline(5,  color='orange', linestyle='--', linewidth=1.5, label='VIF = 5 (moderate)')
ax.set_xlabel('VIF')
ax.set_title('Variance Inflation Factors\n(Red > 10 = severe multicollinearity)')
ax.legend()

plt.suptitle('Assumption N(2) — No Multicollinearity',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('VIF values:')
print(vif_series.to_string())

In [ ]:
# ── Coefficient Instability Under Multicollinearity ────────────────────────
# Demonstration: perturb the data slightly and watch the coefficients
# of TotalRooms and TotalBedrooms change dramatically.

y_log_all = df['LogPrice'].values
X_multi_arr = X_multi.values.astype(float)

# Scale to avoid numerical issues
sc_multi = StandardScaler()
X_multi_sc = sc_multi.fit_transform(X_multi_arr)

# Fit on original data
m_multi = LinearRegression().fit(X_multi_sc, y_log_all)
coefs_original = dict(zip(multi_features, m_multi.coef_))

# Now perturb the data with tiny random noise (1% of each feature's std)
rng = np.random.default_rng(99)
n_perturb = 10
tr_coefs = []      # TotalRooms coefficient across perturbations
tb_coefs = []      # TotalBedrooms coefficient across perturbations

for trial in range(n_perturb):
    noise = rng.normal(0, 0.01, X_multi_arr.shape)   # 1% Gaussian noise
    X_noisy = sc_multi.transform(X_multi_arr + noise * X_multi_arr.std(axis=0))
    m_noisy = LinearRegression().fit(X_noisy, y_log_all)
    noisy_dict = dict(zip(multi_features, m_noisy.coef_))
    tr_coefs.append(noisy_dict['TotalRooms'])
    tb_coefs.append(noisy_dict['TotalBedrooms'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, coefs_list, feat_name in zip(axes,
    [tr_coefs, tb_coefs],
    ['TotalRooms',   'TotalBedrooms']
):
    ax.axhline(coefs_original[feat_name], color='blue', linewidth=2,
               linestyle='--', label=f'Original: {coefs_original[feat_name]:.3f}')
    ax.scatter(range(n_perturb), coefs_list, color='crimson', s=60, zorder=3)
    ax.plot(range(n_perturb), coefs_list, color='crimson', alpha=0.5)
    ax.set_xlabel('Perturbation trial')
    ax.set_ylabel('Coefficient value')
    ax.set_title(f'Coefficient of {feat_name}\nacross 10 tiny data perturbations')
    ax.legend()

plt.suptitle('Multicollinearity → Unstable Coefficients\n'
             '(Small data change = huge coefficient swing)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'TotalRooms coef    original: {coefs_original["TotalRooms"]:.4f}')
print(f'TotalRooms coef    range across perturbations: '
      f'{min(tr_coefs):.4f} to {max(tr_coefs):.4f}')
print(f'TotalBedrooms coef original: {coefs_original["TotalBedrooms"]:.4f}')
print(f'TotalBedrooms coef range   : {min(tb_coefs):.4f} to {max(tb_coefs):.4f}')

In [ ]:
# ── Fix for Multicollinearity: Remove Feature or Use Ridge Regression ───────
# Option 1: Remove TotalBedrooms (keep TotalRooms as it subsumes it)
features_no_tb = [f for f in multi_features if f != 'TotalBedrooms']
X_no_tb = df[features_no_tb].values.astype(float)
sc_no_tb = StandardScaler()
X_no_tb_sc = sc_no_tb.fit_transform(X_no_tb)
vif_no_tb = compute_vif(pd.DataFrame(X_no_tb, columns=features_no_tb))

# Option 2: Ridge regression — penalizes large coefficients, handles multicollinearity
ridge = Ridge(alpha=100)         # alpha controls strength of L2 penalty
ridge.fit(X_multi_sc, y_log_all)
ridge_coefs = dict(zip(multi_features, ridge.coef_))

print('─── After removing TotalBedrooms ───')
print('Top VIFs:')
print(vif_no_tb.head(5).to_string())

print('\n─── Ridge Regression Coefficients (shrinks multicollinear coefs) ───')
print(f'  TotalRooms    (OLS): {coefs_original["TotalRooms"]:.4f}  →  '
      f'(Ridge): {ridge_coefs["TotalRooms"]:.4f}')
print(f'  TotalBedrooms (OLS): {coefs_original["TotalBedrooms"]:.4f}  →  '
      f'(Ridge): {ridge_coefs["TotalBedrooms"]:.4f}')
print('\nRidge shrinks both toward zero, avoiding extreme opposite-sign coefficients.')

---

## Assumption C — No Influential Outliers

A single unusual data point can dominate the OLS fit, tilting the regression plane toward it and away from all other points.  
This was covered in detail in Notebook 09 (Cook's Distance, leverage). Key reminder:

| Measure | What It Detects | Threshold |
|---------|----------------|----------|
| Studentized residual | Unusual **y** (outlier in response) | \|value\| > 3 |
| Leverage ($h_{ii}$) | Unusual **X** (outlier in feature space) | > 2p/n |
| Cook's Distance | Combined influence on all fitted values | > 4/n or > 1 |

**Fix:** Investigate → correct data errors → use robust regression if legitimate outliers exist.

---

## The Gauss-Markov Theorem — A Deeper Look

### Statement

If the following hold:
1. The model is linear in parameters: $y = X\beta + \varepsilon$
2. $E[\varepsilon | X] = 0$ (errors have zero mean conditional on X)
3. $\text{Var}(\varepsilon | X) = \sigma^2 I$ (homoscedasticity + independence)

Then the OLS estimator $\hat{\beta} = (X^TX)^{-1}X^Ty$ is **BLUE** — Best Linear Unbiased Estimator.

### What "Best" Really Means

Among all **linear** and **unbiased** estimators, OLS has the **smallest variance** for every coefficient.  
This means: you cannot build a more precise linear model with the same data — OLS extracts the maximum signal.

### Important Limitations

- "Best" is restricted to **linear unbiased estimators** — a neural network, random forest, or ridge regression can outperform OLS (by accepting some bias to reduce variance)
- **Normality is not required** for BLUE — only linearity, zero mean errors, and homoscedastic independent errors
- Normality *is* required for the t-distributions of hypothesis tests to be exact

### The Bias-Variance Tradeoff at the Estimator Level

$$\text{MSE}(\hat{\beta}) = \underbrace{\text{Variance}(\hat{\beta})}_{\text{OLS minimizes this among linear unbiased}} + \underbrace{\text{Bias}^2(\hat{\beta})}_{= 0 \text{ for OLS}}$$

Ridge regression introduces bias in $\hat{\beta}$ but reduces variance — often winning on overall MSE when there is multicollinearity.

In [ ]:
# ── Gauss-Markov Illustration: OLS is Unbiased ────────────────────────────
# Simulate 1000 datasets from a known DGP, fit OLS each time,
# and verify that the average coefficient equals the true value.

np.random.seed(42)
n_obs      = 200     # observations per dataset
n_trials   = 1000    # number of simulated datasets
true_beta  = np.array([2.0, -1.5, 0.8])   # true coefficients we try to recover

estimated_betas = np.zeros((n_trials, len(true_beta)))

for trial in range(n_trials):
    X_sim = np.random.randn(n_obs, len(true_beta))   # random design matrix
    eps   = np.random.normal(0, 1, n_obs)            # i.i.d. Gaussian errors (GM satisfied)
    y_sim = X_sim @ true_beta + eps                  # DGP: linear + noise

    ols_beta = np.linalg.lstsq(X_sim, y_sim, rcond=None)[0]  # OLS via lstsq
    estimated_betas[trial] = ols_beta

# Visualize distribution of each coefficient across 1000 trials
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for j in range(3):
    ax = axes[j]
    ax.hist(estimated_betas[:, j], bins=50, color='steelblue',
            density=True, alpha=0.7, edgecolor='white')
    ax.axvline(true_beta[j], color='red', linewidth=2, linestyle='-',
               label=f'True β = {true_beta[j]}')
    ax.axvline(estimated_betas[:, j].mean(), color='orange',
               linewidth=2, linestyle='--',
               label=f'Mean β̂ = {estimated_betas[:, j].mean():.4f}')
    ax.set_title(f'OLS Estimates of β_{j+1}\n(across 1000 simulations)')
    ax.set_xlabel('Estimated coefficient')
    ax.legend(fontsize=8)

plt.suptitle('Gauss-Markov in Action: OLS is Unbiased\n'
             '(Red = true value, Orange dashed = mean of 1000 OLS estimates)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('True βs:           ', true_beta)
print('Mean estimated βs: ', estimated_betas.mean(axis=0).round(4))
print('(Should be almost identical — OLS is unbiased under GM conditions)')

---

## Assumptions Checker Function

The following function runs all assumption checks automatically and prints a structured report.  
Use it as a first-pass diagnostic before trusting any regression output.

In [ ]:
# ── Automated Assumptions Checker ──────────────────────────────────────────
def check_linear_regression_assumptions(X, y, feature_names=None, alpha=0.05,
                                         sample_size_shapiro=2000):
    """
    Fit OLS on X, y and run all six assumption checks.
    Prints a structured diagnostic report.

    Parameters
    ----------
    X : array-like, shape (n, p)
    y : array-like, shape (n,)
    feature_names : list of str or None
    alpha : significance level for tests (default 0.05)
    sample_size_shapiro : n to sample for Shapiro-Wilk (valid < ~5000)
    """
    X_arr = np.array(X, dtype=float)
    y_arr = np.array(y, dtype=float)

    if feature_names is None:
        feature_names = [f'X{j}' for j in range(X_arr.shape[1])]

    # Scale before fitting
    sc = StandardScaler()
    X_sc = sc.fit_transform(X_arr)

    # Fit OLS via statsmodels to get full diagnostics
    X_const = sm.add_constant(X_sc)
    sm_ols  = sm.OLS(y_arr, X_const).fit()
    resids  = sm_ols.resid
    fitted  = sm_ols.fittedvalues

    print('=' * 60)
    print('  LINEAR REGRESSION ASSUMPTIONS REPORT')
    print('=' * 60)
    print(f'  n = {len(y_arr)}, p = {X_arr.shape[1]}, R² = {sm_ols.rsquared:.4f}')
    print()

    # ── 1. Linearity (visual proxy via residual correlation with fitted) ─────
    corr_resid_fitted = np.corrcoef(fitted, resids)[0, 1]
    lin_ok = abs(corr_resid_fitted) < 0.1
    print(f'[L] LINEARITY')
    print(f'    Cor(residuals, fitted): {corr_resid_fitted:.4f}')
    print(f'    Status: {"OK" if lin_ok else "WARNING — possible non-linearity"}')
    print()

    # ── 2. Independence (Durbin-Watson) ────────────────────────────────────
    dw = durbin_watson(resids)
    indep_ok = 1.5 < dw < 2.5
    print(f'[I] INDEPENDENCE (Durbin-Watson)')
    print(f'    DW statistic: {dw:.4f}  (target: 1.5 – 2.5)')
    print(f'    Status: {"OK" if indep_ok else "WARNING — possible autocorrelation"}')
    print()

    # ── 3. Normality (Shapiro-Wilk on sample) ──────────────────────────────
    rng2 = np.random.default_rng(42)
    sw_n   = min(sample_size_shapiro, len(resids))
    sw_idx = rng2.choice(len(resids), size=sw_n, replace=False)
    sw_stat, sw_p = stats.shapiro(resids[sw_idx])
    norm_ok = sw_p >= alpha
    print(f'[N] NORMALITY (Shapiro-Wilk, n={sw_n})')
    print(f'    W = {sw_stat:.4f}, p = {sw_p:.6f}')
    print(f'    Status: {"OK" if norm_ok else "WARNING — non-normal residuals"}')
    print(f'    Skewness: {stats.skew(resids):.4f}  |  Kurtosis: {stats.kurtosis(resids):.4f}')
    print()

    # ── 4. Homoscedasticity (Breusch-Pagan) ────────────────────────────────
    _, bp_p, _, _ = het_breuschpagan(resids, sm.add_constant(fitted))
    homo_ok = bp_p >= alpha
    print(f'[E] EQUAL VARIANCE / HOMOSCEDASTICITY (Breusch-Pagan)')
    print(f'    p = {bp_p:.6f}')
    print(f'    Status: {"OK" if homo_ok else "WARNING — heteroscedasticity detected"}')
    print()

    # ── 5. No Multicollinearity (VIF) ──────────────────────────────────────
    X_df = pd.DataFrame(X_sc, columns=feature_names)
    vif_vals = compute_vif(X_df)
    max_vif  = vif_vals.max()
    multi_ok = max_vif < 10
    print(f'[N2] NO MULTICOLLINEARITY (VIF)')
    print(f'    Max VIF: {max_vif:.2f}  ({vif_vals.idxmax()})')
    print(f'    Status: {"OK" if multi_ok else "WARNING — VIF > 10, severe multicollinearity"}')
    print('    Top 3 VIFs:')
    for feat, v in vif_vals.head(3).items():
        flag = ' ← HIGH' if v > 10 else ('' if v <= 5 else ' ← moderate')
        print(f'      {feat:20s}: {v:8.2f}{flag}')
    print()

    # ── 6. No Influential Outliers (Cook's Distance) ────────────────────────
    influence  = sm_ols.get_influence()
    cooks_d    = influence.cooks_distance[0]
    cook_thresh = 4 / len(y_arr)
    n_influen  = (cooks_d > cook_thresh).sum()
    outlier_ok = n_influen < 0.01 * len(y_arr)   # fewer than 1% of points
    print(f'[C] NO INFLUENTIAL OUTLIERS (Cook\'s Distance > 4/n = {cook_thresh:.4f})')
    print(f'    Influential points: {n_influen} ({n_influen/len(y_arr)*100:.1f}% of data)')
    print(f'    Max Cook\'s D: {cooks_d.max():.4f}')
    print(f'    Status: {"OK" if outlier_ok else "WARNING — many influential points"}')
    print()

    # ── Summary ────────────────────────────────────────────────────────────
    statuses = [lin_ok, indep_ok, norm_ok, homo_ok, multi_ok, outlier_ok]
    labels   = ['Linearity', 'Independence', 'Normality', 'Homoscedasticity',
                'No multicollinearity', 'No influential outliers']
    print('─' * 60)
    print('SUMMARY')
    for lbl, ok in zip(labels, statuses):
        status_str = 'PASS' if ok else 'FAIL'
        print(f'  {lbl:30s}: {status_str}')
    n_pass = sum(statuses)
    print(f'\n  {n_pass}/6 assumptions passed.')
    if n_pass == 6:
        print('  Gauss-Markov conditions met — OLS is BLUE for this dataset.')
    else:
        print('  Review warnings above before relying on p-values/CIs.')
    print('=' * 60)

print('check_linear_regression_assumptions() defined.')

In [ ]:
# ── Run the Assumptions Checker on Two Models ──────────────────────────────
print('\n\n>>> MODEL A: Standard features, log price')
check_linear_regression_assumptions(
    X_test_sc, y_test.values,
    feature_names=base_features
)

print('\n\n>>> MODEL B: With multicollinear features added')
X_multi_test = df.loc[y_test.index, multi_features].values
sc_m2 = StandardScaler()
X_multi_test_sc = sc_m2.fit_transform(X_multi_test)
check_linear_regression_assumptions(
    X_multi_test_sc, y_test.values,
    feature_names=multi_features
)

In [ ]:
# ── 6-Panel Summary Diagnostic Figure (One Subplot per Assumption) ─────────
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

X_const_test = sm.add_constant(X_test_sc)
sm_test_fit  = sm.OLS(y_test.values, X_const_test).fit()
resid_t      = sm_test_fit.resid
fitted_t     = sm_test_fit.fittedvalues
infl_t       = sm_test_fit.get_influence()
lev_t        = infl_t.hat_matrix_diag
std_resid_t  = infl_t.resid_studentized_external
cooks_t      = infl_t.cooks_distance[0]

# ── Panel 1: Linearity (Residuals vs Fitted) ───────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(fitted_t, resid_t, alpha=0.2, s=8, color='steelblue')
ax1.axhline(0, color='red', linestyle='--')
lw1 = sm.nonparametric.lowess(resid_t, fitted_t, frac=0.4)
ax1.plot(lw1[:, 0], lw1[:, 1], 'r-', linewidth=2)
ax1.set_title('[L] Linearity\nResiduals vs Fitted (want: flat cloud)')
ax1.set_xlabel('Fitted values')
ax1.set_ylabel('Residuals')

# ── Panel 2: Independence (residuals vs index) ─────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(range(len(resid_t)), resid_t, alpha=0.4, linewidth=0.5, color='seagreen')
ax2.axhline(0, color='red', linestyle='--')
dw_t = durbin_watson(resid_t)
ax2.set_title(f'[I] Independence\nResiduals vs Index — DW = {dw_t:.2f}')
ax2.set_xlabel('Observation index')
ax2.set_ylabel('Residuals')

# ── Panel 3: Normality (Q-Q) ───────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
(osm3, osr3), (sl3, ic3, _) = stats.probplot(resid_t, dist='norm')
ax3.scatter(osm3, osr3, alpha=0.3, s=8, color='mediumpurple')
ax3.plot(osm3, np.array(osm3) * sl3 + ic3, 'r-', linewidth=2)
ax3.set_title('[N] Normality\nQ-Q Plot (want: points on line)')
ax3.set_xlabel('Theoretical quantiles')
ax3.set_ylabel('Sample quantiles')

# ── Panel 4: Homoscedasticity (Scale-Location) ─────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
sqrt_abs4 = np.sqrt(np.abs(resid_t / resid_t.std()))
ax4.scatter(fitted_t, sqrt_abs4, alpha=0.2, s=8, color='darkorange')
lw4 = sm.nonparametric.lowess(sqrt_abs4, fitted_t, frac=0.4)
ax4.plot(lw4[:, 0], lw4[:, 1], 'r-', linewidth=2)
ax4.set_title('[E] Homoscedasticity\nScale-Location (want: flat)')
ax4.set_xlabel('Fitted values')
ax4.set_ylabel('√|Standardized residuals|')

# ── Panel 5: No Multicollinearity (VIF bar chart) ──────────────────────────
ax5 = fig.add_subplot(gs[2, 0])
vif_base = compute_vif(pd.DataFrame(X_test_sc, columns=base_features))
bar_colors5 = ['crimson' if v > 10 else ('orange' if v > 5 else 'steelblue')
               for v in vif_base.values]
ax5.barh(vif_base.index, vif_base.values, color=bar_colors5, alpha=0.8)
ax5.axvline(10, color='red', linestyle='--', linewidth=1.2, label='VIF=10')
ax5.axvline(5,  color='orange', linestyle='--', linewidth=1.2, label='VIF=5')
ax5.set_title('[N2] No Multicollinearity\nVIF (want: all < 5)')
ax5.set_xlabel('VIF')
ax5.legend(fontsize=8)

# ── Panel 6: No Influential Outliers (Cook's D) ────────────────────────────
ax6 = fig.add_subplot(gs[2, 1])
cook_thr6 = 4 / len(resid_t)
bar_colors6 = ['crimson' if d > cook_thr6 else 'steelblue' for d in cooks_t]
ax6.bar(range(len(cooks_t)), cooks_t, color=bar_colors6, width=1, alpha=0.7)
ax6.axhline(cook_thr6, color='red', linestyle='--', linewidth=1.5,
            label=f'Threshold 4/n={cook_thr6:.4f}')
ax6.set_title('[C] No Influential Outliers\nCook\'s Distance (red bars = flagged)')
ax6.set_xlabel('Observation index')
ax6.set_ylabel("Cook's D")
ax6.legend(fontsize=8)

plt.suptitle('6-Panel Linear Regression Assumptions Dashboard',
             fontsize=14, fontweight='bold', y=1.01)
plt.show()

---

## Summary Table: All 6 Assumptions

| # | Assumption | What It Means | How to Check | What Breaks It | Fix |
|---|-----------|---------------|-------------|----------------|-----|
| L | Linearity | X→y relationship is linear in parameters | Scatter plots, residuals vs fitted | Curved relationships, interactions | Polynomial terms, log transforms, interaction features |
| I | Independence | Errors are uncorrelated with each other | Durbin-Watson, residuals vs time | Time series, spatial data, panel data | Lagged features, GLS, ARIMA |
| N | Normality | Errors follow N(0, σ²) | Q-Q plot, Shapiro-Wilk | Skewed targets, outliers | Log-transform target, robust regression |
| E | Equal variance | Var(ε) is constant for all observations | Scale-location plot, Breusch-Pagan | Price data, count data, income data | WLS, log-transform target |
| N2 | No multicollinearity | Features are not perfectly correlated | VIF (>10 = severe) | Derived features (total_rooms + total_bedrooms), one-hot encoding | Remove one feature, PCA, Ridge regression |
| C | No influential outliers | No single point dominates the fit | Cook's Distance, leverage plot | Data entry errors, extreme luxury properties | Investigate, fix errors, robust regression |

---

## Self-Check Questions

<details>
<summary><strong>Q1: Your VIF for <code>total_rooms</code> is 45. What does this mean? Should you remove the feature? What's the alternative?</strong></summary>

**What it means:** VIF = 45 means that 97.8% of the variance in `total_rooms` is explained by the other features in the model ($R^2_j = 1 - 1/45 = 0.978$). The coefficient of `total_rooms` has 45× the variance it would have if the feature were uncorrelated with others — the estimate is extremely imprecise and unstable.

**Should you remove it?** It depends:
- If `total_rooms` and another feature (say `total_bedrooms`) carry duplicate information → remove one. The correlation is a structural issue.
- If `total_rooms` is theoretically important and the correlation is accidental → do not remove.

**Alternatives to removal:**
1. **Ridge regression (L2 penalty):** shrinks the collinear coefficients toward zero, reducing variance without eliminating features
2. **PCA:** combine the collinear features into orthogonal principal components
3. **Domain knowledge:** replace `total_rooms` and `total_bedrooms` with a ratio like `bedroom_fraction = total_bedrooms / total_rooms` — a single feature that captures the relevant signal

</details>

<details>
<summary><strong>Q2: The Gauss-Markov theorem guarantees OLS is BLUE. Does this mean OLS is always the best model overall? What does "linear" in BLUE limit you to?</strong></summary>

**No** — BLUE is the best among a specific, restricted class of estimators.

The word **"linear"** means: estimators of the form $\hat{\beta} = Cy$ where $C$ is a matrix that depends only on $X$ (not on $y$). This is a strong restriction:

- **Not in the competition:** neural networks, random forests, gradient boosting, kernel methods — these are all non-linear estimators and Gauss-Markov says nothing about them.
- **Also not in the competition:** Bayesian shrinkage estimators (like ridge), which are *biased* but can have lower MSE than OLS when features are correlated.

**Practical implication:**
- Gauss-Markov is a theorem about linear models in their ideal conditions.
- In practice, accepting a small bias (ridge, lasso) often reduces variance enough to beat OLS on held-out data.
- For prediction accuracy, use cross-validated model selection. For interpretability with guarantees, use OLS when assumptions hold.

</details>

<details>
<summary><strong>Q3: You have panel data: 100 houses observed monthly for 5 years (500 rows total). Which assumption is most likely violated?</strong></summary>

**Most likely violated: Independence of errors (Assumption I)**

With panel data, there are two sources of dependence:
1. **Serial correlation within each house:** the price of house #42 in Month 2 is highly correlated with its price in Month 1. Errors will cluster by time within each house.
2. **Cross-sectional correlation:** all houses in the same market are affected by the same macroeconomic shocks (interest rate changes, recessions) → errors in Month 1 are correlated across all 100 houses simultaneously.

**What this means:** the effective sample size is much less than 500. Standard errors computed as if errors are independent will be severely under-estimated.

**Fixes:**
- Use **fixed effects** model: add house-level dummies to absorb time-invariant characteristics
- Use **clustered standard errors**: correct standard errors for within-cluster correlation (common in economics)
- Use **panel-data models**: `statsmodels.PanelOLS`, or random effects GLS
- Use **time-series features:** lag of price, month dummies, year trend

</details>

<details>
<summary><strong>Q4: Normality of errors is violated, but your sample size is large (n = 10,000). Does this still matter for predictions? What about for hypothesis tests on coefficients?</strong></summary>

**For predictions (point estimates):** Normality violation does **not matter** for predictions.  
OLS gives unbiased coefficients under the Gauss-Markov theorem without requiring normality. Your point predictions $\hat{y}$ are still valid.

**For hypothesis tests and confidence intervals:**
- With n = 10,000, the **Central Limit Theorem** rescues you. The sampling distribution of $\hat{\beta}$ converges to normal even if individual errors are not normal — by a large-sample argument.
- t-tests and Wald tests remain approximately valid for large n.
- **Exception:** if normality is violated due to **heavy-tailed errors** (not just skewness), extreme outliers can still distort results even with large n.

**Practical advice:**
- n = 10,000 with moderate non-normality (skewness) → hypothesis tests are approximately fine, use them with awareness
- n = 10,000 with extreme outliers → Cook's distance matters; use robust regression or investigate
- n = 30 with strong non-normality → switch to bootstrap confidence intervals or Bayesian methods

</details>